# Project 17 — Mini LLM Server (FastAPI + streaming + batching + queue + metrics)

```
TinyGPT (L2/d64, brief CPU train, char-level)
  -> FastAPI app: GET /health | POST /generate | POST /generate_stream (SSE) | POST /batch
  -> bounded in-flight queue (semaphore, 429 when full) + metrics (counts, TTFT proxy, tok/s)
  -> all tests via fastapi.testclient.TestClient (no ports, offline-safe; `uvicorn notebook: app` pattern noted below)
  -> bench: single vs batch throughput + streaming chunks + threaded concurrency + queue-overflow ablation
```

Goal: learn the serving slice that P13–P16 never touch — framing, batching, backpressure,
and observability — on a real HTTP app you can `curl`.

Learning objectives:
1. Sketch request path: parse -> tokenize -> prefill -> decode loop -> detokenize -> metrics.
2. Explain why streaming (TTFT) matters separately from throughput (tok/s).
3. Show batching amortizes prefill; show bounded queues turn overload into 429s instead of OOMs.
4. Read server metrics and size `max_inflight` / `max_tokens` for a latency SLO.
   Key lesson: batching buys throughput, streaming buys perceived latency, queues buy survival.


In [ ]:
import os # need os to make results/ and read cpu count
import random # need random to seed stdlib RNG
import time # need time to measure TTFT + per-request latency
import csv # need csv to write server.csv + batch sweep csv
import math # need math for ln(V) baseline + perplexity
import json as js # need json to parse SSE payload frames in tests
import queue # need queue for the bounded admission queue demo (what-if)
import threading # need threading for the concurrency/overload test
import urllib.request # need urllib to try downloading TinyShakespeare (offline fallback)
import numpy as np # need numpy for batch indexing + median stats
import torch # need torch for the TinyGPT being served
import torch.nn.functional as F # need F for cross-entropy during the brief train
import matplotlib # need matplotlib for throughput figures
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for batch + queue figures
from fastapi import FastAPI, HTTPException # need FastAPI for the app + 429/400 errors
from fastapi.responses import StreamingResponse, JSONResponse # need StreamingResponse for SSE token stream
from fastapi.testclient import TestClient # need TestClient to test the app with no ports/network
SEED = 17 # fixed seed: same weights + same prompts every run
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing on shared boxes
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU
print(f"device={DEV} torch={torch.__version__}") # env line for the README


In [ ]:
# ---- data + model: same TinyShakespeare/char recipe as P16, brief train ----
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' # same canonical mirror as P13-P16
try: # try the download first
    raw, _ = urllib.request.urlretrieve(URL, './results/tinyshakespeare.txt') # download into results/ (cache)
    text = open(raw, encoding='utf-8').read() # read it
    print(f"downloaded chars={len(text)}") # confirm live corpus
except Exception as e: # offline fallback mirrors P16 exactly
    print(f"download failed ({e}); using synthetic fallback") # say why
    rng = np.random.RandomState(SEED) # seeded RNG: same fallback every run
    words = ['king', 'queen', 'thou', 'thee', 'night', 'light', 'love', 'hate', 'fire', 'water'] # learnable pseudo-vocab
    text = ' '.join(rng.choice(words, size=60000).tolist()) # 60k pseudo-words
    print(f"synthetic chars={len(text)}") # confirm fallback
chars = sorted(set(text)) # char vocab
V = len(chars) # vocab size
stoi = {c: i for i, c in enumerate(chars)} # encode map
itos = {i: c for c, i in stoi.items()} # decode map
data = torch.tensor([stoi[c] for c in text], dtype=torch.long) # corpus ids
CTX = 64 # serving context cap (prompts truncate here; keeps CPU fast)
class Block(torch.nn.Module): # one pre-norm transformer block (same as P16)
    def __init__(self, d, h): # d width, h heads
        super().__init__() # bookkeeping
        self.ln1 = torch.nn.LayerNorm(d) # pre-attn norm
        self.attn = torch.nn.MultiheadAttention(d, h, batch_first=True) # stock MHA
        self.ln2 = torch.nn.LayerNorm(d) # pre-mlp norm
        self.fc1 = torch.nn.Linear(d, 4 * d) # mlp expand
        self.fc2 = torch.nn.Linear(4 * d, d) # mlp project
    def forward(self, x): # (b, t, d) -> (b, t, d)
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), need_weights=False) # self-attn
        x = x + a # residual 1
        return x + self.fc2(torch.nn.functional.gelu(self.fc1(self.ln2(x)))) # residual 2
class TinyGPT(torch.nn.Module): # char-level GPT
    def __init__(self, L, d, h, v, ctx): # dims
        super().__init__() # bookkeeping
        self.tok = torch.nn.Embedding(v, d) # token table
        self.pos = torch.nn.Embedding(ctx, d) # positions
        self.blocks = torch.nn.ModuleList([Block(d, h) for _ in range(L)]) # stack
        self.ln = torch.nn.LayerNorm(d) # final norm
        self.head = torch.nn.Linear(d, v, bias=False) # LM head
    def forward(self, x): # (b, t) -> (b, t, v)
        b, t = x.shape # read dims
        h = self.tok(x) + self.pos(torch.arange(t, device=x.device)) # embed
        for blk in self.blocks: # stack
            h = blk(h) # block
        return self.head(self.ln(h)) # logits
model = TinyGPT(2, 64, 4, V, CTX).to(DEV) # tiny served model (~110k params)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4) # brief-train optimizer
for it in range(60): # brief train so generations are not uniform noise (seconds on CPU)
    ix = torch.randint(0, len(data) - CTX - 1, (32,)) # random windows
    xb = torch.stack([data[i:i + CTX] for i in ix]).to(DEV) # inputs
    yb = torch.stack([data[i + 1:i + CTX + 1] for i in ix]).to(DEV) # shifted labels
    loss = F.cross_entropy(model(xb).view(-1, V), yb.view(-1)) # next-token CE
    opt.zero_grad() # clear
    loss.backward() # backprop
    opt.step() # update
print(f"served model trained: last_loss={loss.item():.4f} (init~{math.log(V):.2f})") # training card
def encode(s): # prompt string -> id list, truncated to CTX-1 (leave room to generate)
    return [stoi.get(c, 0) for c in s[:CTX - 1]] # unknown chars map to id 0 (explicit, documented)
def decode(ids): # id list -> string
    return ''.join(itos[i] for i in ids) # map each id back
def greedy_continue(ids, k): # greedy decode k tokens after ids (single request core)
    model.eval() # eval mode
    out = list(ids) # copy prompt ids
    with torch.no_grad(): # no graph for serving
        for _ in range(k): # k steps
            x = torch.tensor([out[-CTX:]], dtype=torch.long).to(DEV) # last-CTX window (1, t)
            out.append(model(x)[0, -1].argmax().item()) # argmax next id
    model.train() # restore train mode
    return out[len(ids):] # only the new ids


In [ ]:
# ---- app: routes + bounded admission + metrics (the serving slice) ----
app = FastAPI(title='mini-llm-server (P17)') # the ASGI app TestClient will drive
MAX_INFLIGHT = 4 # admission bound: >4 concurrent generates get 429 (backpressure, not OOM)
MAX_TOKENS = 64 # per-request cap: bounds worst-case decode latency
_sema = threading.Semaphore(MAX_INFLIGHT) # counting semaphore for in-flight slots
MET = {'requests': 0, 'stream_requests': 0, 'batch_requests': 0, 'rejected_429': 0, # lifetime counters
       'lat_ms': [], 'ttft_ms': [], 'tok_s': []} # per-request latency/TTFT/throughput samples
def _metric(lat, ttft, k): # record one completed non-stream request
    MET['lat_ms'].append(lat) # end-to-end ms
    MET['ttft_ms'].append(ttft) # prefill-to-first-token ms (non-stream: same as first decode step)
    MET['tok_s'].append(k / max(lat / 1e3, 1e-6)) # tokens per second
@app.get('/health') # liveness probe (load balancer / k8s style)
def health(): # no model touch: must stay fast
    return {'status': 'ok', 'device': DEV, 'vocab': V} # status + env card
@app.post('/generate') # core endpoint: prompt -> text
def generate(body: dict): # body: {"prompt": str, "max_tokens": int}
    prompt = str(body.get('prompt', '')) # coerce prompt to str
    k = min(int(body.get('max_tokens', 16)), MAX_TOKENS) # clamp output length (SLO guard)
    if not prompt: # empty prompt is a client error, not a model call
        raise HTTPException(400, 'empty prompt') # 400 with reason
    if not _sema.acquire(blocking=False): # try to take a slot without waiting (shed fast)
        MET['rejected_429'] += 1 # count the shed
        raise HTTPException(429, 'server busy (in-flight limit)') # 429 tells the client to retry
    try: # guaranteed release path below
        t0 = time.time() # request start
        ids = encode(prompt) # tokenize (prompt side)
        t1 = time.time() # prefill start proxy (first-token timer starts here)
        new = greedy_continue(ids, k) # decode k tokens greedily
        t2 = time.time() # request end
        MET['requests'] += 1 # count success
        _metric((t2 - t0) * 1e3, (t2 - t1) * 1e3 / max(k, 1) + (t1 - t0) * 1e3, k) # lat + TTFT-proxy + tok/s
        return {'text': decode(new), 'tokens': k, 'lat_ms': round((t2 - t0) * 1e3, 1)} # text + timings
    finally: # always free the slot
        _sema.release() # release admission slot
@app.post('/generate_stream') # SSE endpoint: same decode, chunked as data: frames
def generate_stream(body: dict): # body: same shape as /generate
    prompt = str(body.get('prompt', '')) # coerce prompt
    k = min(int(body.get('max_tokens', 16)), MAX_TOKENS) # clamp output length
    if not prompt: # validate before streaming headers
        raise HTTPException(400, 'empty prompt') # 400, same contract
    def gen(): # generator of SSE frames (one token per frame + done marker)
        MET['stream_requests'] += 1 # count the stream
        ids = list(encode(prompt)) # mutable id buffer (prompt + generated)
        model.eval() # eval mode for the stream
        with torch.no_grad(): # no graph
            for i in range(k): # k token steps
                x = torch.tensor([ids[-CTX:]], dtype=torch.long).to(DEV) # window
                nxt = model(x)[0, -1].argmax().item() # greedy next id
                ids.append(nxt) # extend buffer (KV-cache would avoid recompute; P18 quantifies that)
                yield f"data: {js.dumps({'i': i, 'token': itos[nxt]})}\n\n" # one SSE frame per token
        model.train() # restore train mode
        yield "data: [DONE]\n\n" # terminal frame clients watch for
    return StreamingResponse(gen(), media_type='text/event-stream') # chunked response
@app.post('/batch') # batched endpoint: N prompts share one prefill (amortized)
def batch_gen(body: dict): # body: {"prompts": [str], "max_tokens": int}
    prompts = body.get('prompts', []) # prompt list
    k = min(int(body.get('max_tokens', 16)), MAX_TOKENS) # shared output cap
    if not prompts or len(prompts) > 16: # guard empty + absurd batches
        raise HTTPException(400, 'need 1..16 prompts') # 400 with contract
    if not _sema.acquire(blocking=False): # batch takes one slot (coarse but explicit)
        MET['rejected_429'] += 1 # count shed
        raise HTTPException(429, 'server busy (in-flight limit)') # same 429 contract
    try: # guaranteed release
        t0 = time.time() # batch start
        ids = [encode(p) for p in prompts] # tokenize all prompts
        L = max(len(i) for i in ids) # pad target = longest prompt
        pad = torch.stack([torch.tensor(i + [0] * (L - len(i)), dtype=torch.long) for i in ids]).to(DEV) # (B, L) padded prefill
        model.eval() # eval mode
        with torch.no_grad(): # no graph
            _ = model(pad) # one shared prefill forward (the amortization win)
            outs = [] # per-prompt completions
            for prompt_ids in ids: # greedy decode per sequence (decode still per-row here; continuous batching in P18 interleaves)
                outs.append(decode(greedy_continue(prompt_ids, k))) # reuse the single core
        model.train() # restore train mode
        dt = (time.time() - t0) * 1e3 # batch ms
        MET['batch_requests'] += 1 # count batch
        return {'texts': outs, 'tokens_each': k, 'batch_ms': round(dt, 1), # texts + timing
                'tok_s': round(len(prompts) * k / max(dt / 1e3, 1e-6), 1)} # batch throughput
    finally: # always free
        _sema.release() # release slot
@app.get('/metrics') # observability endpoint (Prometheus-style thinking, JSON here)
def metrics(): # aggregate the MET buffers into medians
    def med(a): # median-or-zero helper (empty at startup)
        return round(float(np.median(a)), 2) if a else 0.0 # median of samples
    return {'requests': MET['requests'], 'stream_requests': MET['stream_requests'], # counters
            'batch_requests': MET['batch_requests'], 'rejected_429': MET['rejected_429'], # counters
            'med_lat_ms': med(MET['lat_ms']), 'med_ttft_ms': med(MET['ttft_ms']), # latency medians
            'med_tok_s': med(MET['tok_s'])} # throughput median


In [ ]:
# ---- tests + benchmarks: correctness first, then numbers (all via TestClient) ----
cli = TestClient(app) # in-process client: no ports, no network, deterministic
r = cli.get('/health') # liveness probe
assert r.status_code == 200 and r.json()['status'] == 'ok' # health must pass
print('health:', r.json()) # show the card
r = cli.post('/generate', json={'prompt': 'the king', 'max_tokens': 8}) # basic generation
assert r.status_code == 200 and len(r.json()['text']) == 8 # 8 chars back (char-level)
print('generate:', r.json()) # show one response
r = cli.post('/generate', json={'prompt': '', 'max_tokens': 8}) # empty prompt contract
assert r.status_code == 400 # must be 400, not 500
r = cli.post('/generate', json={'prompt': 'hi', 'max_tokens': 1000}) # over-cap clamp
assert r.status_code == 200 and r.json()['tokens'] == MAX_TOKENS # clamped to the cap
print('contracts ok (400 empty, clamp max_tokens)') # confirm API contracts
r = cli.post('/generate_stream', json={'prompt': 'night', 'max_tokens': 8}) # SSE smoke test
frames = [ln for ln in r.text.splitlines() if ln.startswith('data:')] # SSE data lines only
assert len(frames) == 9 and frames[-1] == 'data: [DONE]' # 8 tokens + DONE marker
first = js.loads(frames[0][len('data: '):]) # parse first frame
assert set(first) == {'i', 'token'} # frame schema check
print(f"stream: {len(frames) - 1} token frames + DONE") # confirm streaming
r = cli.post('/batch', json={'prompts': ['king', 'queen', 'night'], 'max_tokens': 8}) # batch smoke test
assert r.status_code == 200 and len(r.json()['texts']) == 3 # 3 texts back
print('batch:', {k: r.json()[k] for k in ('batch_ms', 'tok_s')}) # show batch timing
# -- bench: single (sequential) vs batch (one prefill) at equal token budgets --
def bench_single(n_req=8, k=16): # sequential /generate calls
    t0 = time.time() # wall start
    for i in range(n_req): # one HTTP call each (no client-side batching)
        rr = cli.post('/generate', json={'prompt': f"prompt {i} thou", 'max_tokens': k}) # fixed prompt shape
        assert rr.status_code == 200 # every call must succeed (slots free sequentially)
    dt = time.time() - t0 # wall seconds
    return dt, n_req * k / dt # (seconds, tok/s)
def bench_batch(n_req=8, k=16): # one /batch call for all prompts
    t0 = time.time() # wall start
    rr = cli.post('/batch', json={'prompts': [f"prompt {i} thou" for i in range(n_req)], 'max_tokens': k}) # single call
    dt = time.time() - t0 # wall seconds
    assert rr.status_code == 200 # must succeed
    return dt, n_req * k / dt # (seconds, tok/s)
s_dt, s_tps = bench_single() # sequential baseline
b_dt, b_tps = bench_batch() # batched contender
print(f"single: {s_dt:.2f}s {s_tps:.0f} tok/s | batch8: {b_dt:.2f}s {b_tps:.0f} tok/s (x{b_tps / max(s_tps, 1e-6):.2f})") # headline comparison
rows = [{'mode': 'single_x8', 'seconds': round(s_dt, 3), 'tok_s': round(s_tps, 1)}, # sequential row
        {'mode': 'batch_8', 'seconds': round(b_dt, 3), 'tok_s': round(b_tps, 1)}] # batch row
# -- concurrency + backpressure: 8 threads vs 4 slots -> some 429s, zero 500s --
codes = [] # status codes across threads
def hit(i): # one threaded request (own client per thread: TestClient portals are not shared)
    try: # capture codes, never raise
        rr = TestClient(app).post('/generate', json={'prompt': f"race {i}", 'max_tokens': 16}) # same endpoint
        codes.append(rr.status_code) # record 200 or 429
    except Exception: # any transport error counts distinctly
        codes.append(-1) # -1 = client-side error (must be zero)
ts = [threading.Thread(target=hit, args=(i,)) for i in range(8)] # 8 racers, 4 slots
[t.start() for t in ts] # launch all
[t.join() for t in ts] # wait all
n200, n429 = codes.count(200), codes.count(429) # successes vs shed
assert n200 + n429 == 8 and -1 not in codes # only 200/429 allowed (no 500s, no transport errors)
print(f"concurrency: 8 threads -> {n200}x200 + {n429}x429 (backpressure, no 500s)") # overload card
rows.append({'mode': 'concurrent_8', 'seconds': -1, 'tok_s': -1}) # placeholder row keeps schema (real signal is the line above)
m = cli.get('/metrics').json() # read the metrics endpoint
print('metrics:', m) # show counters + medians
assert m['requests'] >= 8 # at least the bench singles went through
with open('./results/server.csv', 'w', newline='') as f: # save the bench table
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(rows) # one row per mode
print('saved ./results/server.csv') # confirm artifact


In [ ]:
# ---- what-if: batch-size sweep (1,2,4,8) + queue-depth ablation + figure ----
sweep = [] # batch-size sweep rows (same k, more sharing)
for B in [1, 2, 4, 8]: # batch widths: prefill amortization should flatten per-prompt cost
    t0 = time.time() # wall start
    rr = cli.post('/batch', json={'prompts': [f"sweep {i} night" for i in range(B)], 'max_tokens': 16}) # one call, B prompts
    dt = time.time() - t0 # wall seconds
    assert rr.status_code == 200 # must succeed
    sweep.append({'B': B, 'seconds': round(dt, 3), 'tok_s': round(B * 16 / max(dt, 1e-6), 1), # throughput row
                  'ms_per_prompt': round(dt * 1e3 / B, 1)}) # per-prompt cost (should fall with B)
    print(sweep[-1]) # one line per width
assert sweep[-1]['ms_per_prompt'] < sweep[0]['ms_per_prompt'] # batching must cut per-prompt cost (else prefill is trivial)
qlab = [] # queue-depth ablation: bounded queue as an admission model (no threads needed)
for maxq, arrivals in [(4, 8), (64, 8)]: # small bound sheds; big bound absorbs (same 8 arrivals)
    q = queue.Queue(maxsize=maxq) # bounded FIFO of admitted jobs
    shed = 0 # rejected count
    for i in range(arrivals): # 8 instant arrivals (burst)
        try: # non-blocking put
            q.put_nowait(i) # admit if room
        except queue.Full: # bound hit
            shed += 1 # shed like a 429
    qlab.append({'max_queue': maxq, 'arrivals': arrivals, 'shed': shed, # identity + shed columns
                 'served': arrivals - shed}) # served column
    print(qlab[-1]) # one line per bound
assert qlab[0]['shed'] > qlab[1]['shed'] # small queue must shed more (sanity on the model)
with open('./results/batch_sweep.csv', 'w', newline='') as f: # save batch sweep
    w = csv.DictWriter(f, fieldnames=list(sweep[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(sweep) # one row per width
print('saved ./results/batch_sweep.csv') # confirm artifact
fig, ax = plt.subplots(1, 2, figsize=(11, 4)) # two panels: batch scaling + queue shedding
ax[0].plot([s['B'] for s in sweep], [s['tok_s'] for s in sweep], marker='o') # throughput vs width
ax[0].set_xlabel('batch width B') # x label
ax[0].set_ylabel('tok/s') # y label
ax[0].set_title('P17: batching throughput') # panel title
ax[0].grid(True, alpha=0.3) # light grid
ax[1].bar([f"q{q['max_queue']}" for q in qlab], [q['shed'] for q in qlab]) # shed bars per bound
ax[1].set_xlabel('queue bound') # x label
ax[1].set_ylabel('shed requests (of 8)') # y label
ax[1].set_title('P17: bounded queue sheds bursts') # panel title
fig.tight_layout() # avoid overlap
fig.savefig('./results/server.png', dpi=120) # save figure
print('saved ./results/server.png') # confirm artifact


Cleanup / next steps:
- Artifacts: `results/server.csv` (single vs batch), `results/batch_sweep.csv` (B=1..8),
  `results/server.png`, `results/tinyshakespeare.txt` (cache when online).
- To serve for real: `uvicorn notebook:app --port 8000` (app object is named `app`;
  endpoints `/health`, `/generate`, `/generate_stream`, `/batch`, `/metrics` all `curl`-able).
- Limits of this lab: greedy decode only, per-row decode loop, no KV-cache, no continuous
  batching — P18 simulates exactly those scheduler upgrades and quantifies TTFT/TPOT wins.
